# Convert Benchmark (PFF→L0 Zarr)

Measures PFF read throughput, ZarrPythonWriter write speed, checksum overhead, sharding effect on file count, codec/level trade-offs.

In [ ]:
from pathlib import Path

from panoseti_analysis.io.bench import BenchResult, stage_timer, summarize
from panoseti_analysis.paths import REPO_ROOT

# ── Configure ───────────────────────────────────────────────────────────────────────────────
OBS_DIR = Path("/path/to/obs.pffd")  # replace with a real .pffd run
OUT_BASE = Path("/tmp/convert_bench")

results: list[BenchResult] = []

## §1 Baseline: ZarrPythonWriter, no checksum, no sharding

In [ ]:
from panoseti_analysis.adapters.convert import run_convert

OUT_DIR = OUT_BASE / "baseline"
out_bytes = 0  # fill after run

with stage_timer("convert_baseline", bytes_in=0) as r:
    records = run_convert(OBS_DIR, OUT_DIR, checksum=False)
    out_bytes = sum(f.stat().st_size for f in OUT_DIR.rglob("*") if f.is_file())

r.bytes_out = out_bytes
results.append(r)
print(summarize(results))

## §2 With checksum

In [ ]:
OUT_DIR = OUT_BASE / "with_checksum"
out_bytes = 0  # fill after run

with stage_timer("convert_with_checksum", bytes_in=0) as r:
    records = run_convert(OBS_DIR, OUT_DIR, checksum=True)
    out_bytes = sum(f.stat().st_size for f in OUT_DIR.rglob("*") if f.is_file())

r.bytes_out = out_bytes
results.append(r)
print(summarize(results))

## §3 With sharding (shard_factor=16)

In [ ]:
OUT_DIR = OUT_BASE / "sharded"
out_bytes = 0  # fill after run

with stage_timer("convert_sharded", bytes_in=0) as r:
    records = run_convert(OBS_DIR, OUT_DIR, checksum=False, shard_factor=16)
    out_bytes = sum(f.stat().st_size for f in OUT_DIR.rglob("*") if f.is_file())

r.bytes_out = out_bytes
results.append(r)
print(summarize(results))

## §4 File count comparison

In [ ]:
for label, out_dir in [("baseline", OUT_BASE / "baseline"), ("sharded", OUT_BASE / "sharded")]:
    files = list(out_dir.rglob("*"))
    print(f"{label}: {len(files)} files")